# Tarea 07: Una cuenta en un aro que gira

**Semana 7** · Entrega: antes de la clase de la Semana 8

## Objetivos de la tarea

1. Orientar marcos de referencia encadenados, con ángulos que dependen del
   tiempo, y leer sus matrices de cosenos directores.
2. Obtener la posición, velocidad y aceleración de un punto con `Point`, y
   verificarlas con el teorema de dos puntos.
3. Interpretar físicamente los términos de la rapidez y de la aceleración.

In [ ]:
import sympy as sp
import sympy.physics.mechanics as me
from IPython.display import Math

me.init_vprinting()

# Ya declarados: no los cambies, los tests los usan tal cual.
t = sp.Symbol("t")               # el tiempo de mechanics: sin suposiciones
N = me.ReferenceFrame("N")       # el marco del laboratorio
centro = me.Point("O")           # el centro del aro
centro.set_vel(N, 0)             # fijo en el laboratorio

## El sistema

Un aro de alambre de radio $R$ está parado verticalmente y **gira** alrededor
de su diámetro vertical con velocidad angular constante $\Omega$. Una cuenta
ensartada en el alambre puede deslizarse por él; su posición sobre el aro la da
el ángulo $\theta(t)$, medido desde el punto más bajo.

![Aro de radio R que gira con Omega alrededor del eje vertical n_z; la cuenta esta a un angulo theta del punto mas bajo, del lado de -a_x](img/cuenta-aro.svg)

Lo describimos con tres marcos encadenados:

- `N`: el laboratorio, con $\hat{n}_z$ vertical hacia arriba.
- `A`: el **aro**. Es $N$ girado un ángulo $\Omega t$ alrededor de `N.z`, así
  que el plano del aro es el plano $\hat{a}_x$–$\hat{a}_z$.
- `B`: la **cuenta**. Es $A$ girado un ángulo $\theta$ alrededor de `A.y`, el
  eje perpendicular al plano del aro (en el dibujo, entra en la página).

En este arreglo la cuenta está en $-R\,\hat{b}_z$: en $\theta = 0$ eso es el
punto más bajo del aro, justo debajo del centro.

Todo lo que necesitas está en los dos notebooks de la semana 7.

## Ejercicio 1: símbolos y marcos

Completa la celda con estos nombres exactos, que son los que usa
`tests/test_tarea.py`:

- `radio` y `omega`: símbolos **positivos**, con nombres de símbolo `"R"` y
  `"Omega"` — un radio y una rapidez de giro no son negativos.
- `theta`: el ángulo de la cuenta, como `dynamicsymbol` de nombre `"theta"`.
- `A` y `B`: los marcos del aro y de la cuenta, orientados como dice la
  descripción del sistema, con `orient_axis`.

Antes de seguir, muestra `A.dcm(N)` y `B.dcm(A)` y compáralos con la DCM de la
sesión 1. Una de las dos no tiene la misma forma que la de la clase: ¿por qué?

In [ ]:
# TODO: completa el ejercicio 1
radio = sp.Symbol('R', positive=True)
omega = sp.Symbol('Omega', positive=True)
theta = me.dynamicsymbols('theta')

A = me.ReferenceFrame('A')

B = me.ReferenceFrame('B')

A.orient_axis(N, omega*t,N.z)
B.orient_axis(A, theta, A.y)

display(sp.simplify(A.dcm(N)))
display(sp.simplify(B.dcm(A)))

Es diferente porque están girando sobre ejes distintos. Al rotar el eje Y, el '1' queda en el centro de la matriz y los senos y cosenos se acomodan en las otras posiciones.

## Ejercicio 2: posición, velocidad y aceleración

- `cuenta`: un `me.Point` ubicado respecto a `centro` en $-R\,\hat{b}_z$.
- `posicion`: la posición de la cuenta respecto a `centro`.
- `velocidad` y `aceleracion`: su velocidad y su aceleración **en `N`**.

Las expresiones en `N` salen largas. Para leerlas, muéstralas con `express(B)`,
en la base de la cuenta: se simplifican mucho. Etiquétalas con `Math` y
`me.vlatex`, como en clase.

In [ ]:
# TODO: completa el ejercicio 2
cuenta = me.Point('cuenta')
cuenta.set_pos(centro,-radio*B.z)

posicion = cuenta.pos_from(centro)

velocidad = cuenta.vel(N)

aceleracion = cuenta.acc(N)

display(Math(r'\mathbf{r} = ' + me.vlatex(posicion.express(B))))
display(Math(r'\mathbf{v} = ' + me.vlatex(velocidad.express(B))))
display(Math(r'\mathbf{a} = ' + me.vlatex(aceleracion.express(B))))

## Ejercicio 3: velocidad angular y teorema de transporte

- `velocidad_angular`: la velocidad angular de `B` en `N`. Debe tener dos
  pedazos: el giro del aro y el deslizamiento de la cuenta.
- `velocidad_transporte`: $\boldsymbol{\omega}^B \times \mathbf{r}$, el producto
  cruz de `velocidad_angular` con `posicion`.

`velocidad_transporte` tiene que coincidir con la `velocidad` del ejercicio 2.
Compruébalo tú: la diferencia, expresada en `N` y simplificada, debe ser cero.
¿Qué teorema de la sesión 2 estás usando, y por qué aplica aquí aunque la
cuenta **sí** se mueve sobre el aro?

In [ ]:
# TODO: completa el ejercicio 3
velocidad_angular = omega * N.z+theta.diff(t)*A.y

velocidad_transporte = velocidad_angular.cross(posicion)

diff= (velocidad_transporte-velocidad).express(N)

print('Diferencia:')
display(sp.simplify(diff.to_matrix(N)))

Estamos usando el Teorema del Transporte. Y aplica aquí aunque la cuenta se mueva, porque el vector $r$ que definimos previamente es un vector que está fijo en el marco B. Y como el marco B gira junto con la cuenta, la derivada de $r$ vista desde B es cero.

## Ejercicio 4: rapidez y aceleración a lo largo del aro

- `rapidez_cuadrada`: $\mathbf{v} \cdot \mathbf{v}$, simplificada con
  `sp.simplify`. Es un escalar: etiquétalo con `sp.Eq`.
- `aceleracion_tangencial`: la componente de `aceleracion` a lo largo de
  $\hat{b}_x$, simplificada.

Todo con cantidades exactas: nada de decimales.

In [ ]:
# TODO: completa el ejercicio 4
rapidez_cuadrada = sp.simplify(velocidad.dot(velocidad))

aceleracion_tangencial = sp.simplify(aceleracion.dot(B.x))

display(sp.Eq(sp.Symbol('v^2'), rapidez_cuadrada))
display(sp.Eq(sp.Symbol('a_x'), aceleracion_tangencial))

## Discusión

_Contesta aquí, en uno o dos párrafos. Esta parte no la califica el
autograding: la revisa el asistente._

1. `rapidez_cuadrada` tiene dos términos. ¿Cuál viene de que la cuenta se
   desliza por el aro y cuál de que el aro gira? ¿Por qué el segundo se anula en
   $\theta = 0$?
2. En `aceleracion_tangencial` aparece $-R\ddot\theta$, con signo menos. ¿Por
   qué? (Pista: ¿hacia dónde apunta $\hat{b}_x$ cuando $\theta$ crece?)
3. El término con $\Omega^2$ de `aceleracion_tangencial` es el que, visto desde
   el aro, se siente como la "fuerza centrífuga" que empuja a la cuenta hacia
   afuera. ¿En qué valores de $\theta$ se anula, y por qué tiene sentido
   físicamente en cada uno?

_(Escribe tu respuesta aquí.)_

1. Vemos que la rapidez cuadrada viene dada por la expresión: $R^2 \dot{\theta} + R^2 \Omega^2 \sin(\theta)^2 $. El primer término corresponde a la velocidad de la cuenta moviendose a lo largo del aro, depende de $\dot{\theta}$. El segundo término viene de la rotación del aro y depende de $\Omega$. Este término se anula en $\theta = 0$ porque en ese punto la cuenta se encuentra sobre el eje de rotación vertical. Su distancia al eje es $R \sin(0) = 0$, por lo que el giro del aro no le produce ninguna velocidad tangencial.

2. El vector unitario $b_x$ apunta en la dirección en la que disminuye $\theta$. Entonces, si la cuenta acelera aumentando su ángulo $\theta$, su aceleración tangencial va en dirección contraria al eje $b_x$ (en la dirección del vector unitario).

3. El término que acompaña a $\Omega^2$ se anula en $\theta = 0$ y en $\theta = \frac{\pi}{2}$.

En $\theta = 0$, tiene sentido porque la cuenta está sobre el eje de rotación. Su radio de giro es cero, por lo que no siente fuerza centrífuga.

En $\theta = \frac{\pi}{2}$, la cuenta está en el punto más alejado del eje, por lo que la fuerza centrífuga es máxima, pero apunta horizontalmente hacia afuera. Como la tangente del aro en ese punto es vertical, la fuerza centrífuga es perpendicular al movimiento de la cuenta a lo largo del aro, por lo que su componente tangancial es cero.

## Resumen

Entrega esta tarea por **Pull Request dentro de tu fork**, como las anteriores.
El paso a paso está en [`docs/git-guia.md`](../../docs/git-guia.md), y el
checklist completo en [`README.md`](README.md).

Antes de entregar, corre el notebook desde un kernel limpio
(`Kernel → Restart & Run All`) y confirma que llega hasta el final sin errores.
Al hacer push, GitHub Actions corre los tests y deja un check ✅ o ❌ en tu
Pull Request.